### Step 1: Load the source data

In [0]:
incremental_silver_orders = spark.table(
    "workspace.ecommerce_project.incremental_silver_orders"
)

### Step 2: Load the target Delta table

In [0]:
from delta.tables import DeltaTable

silver_target = DeltaTable.forName(
    spark,
    "workspace.ecommerce_project.silver_orders"
)

In [0]:
(
    silver_target.alias("target")
    .merge(
        incremental_silver_orders.alias("source"),
        "target.order_id = source.order_id" 
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
spark.sql("""
          select order_id,order_status,payment_status,delivery_date
          from workspace.ecommerce_project.silver_orders
          order by order_id
          """).show()

+---------+------------+--------------+-------------+
| order_id|order_status|payment_status|delivery_date|
+---------+------------+--------------+-------------+
|ORD100001|   Delivered|          PAID|   2025-03-07|
|ORD100002|     Shipped|          PAID|         NULL|
|ORD100003|    Returned|          PAID|   2025-12-29|
|ORD100004|   Confirmed|          PAID|         NULL|
|ORD100005|   Confirmed|       PENDING|         NULL|
|ORD100006|   Delivered|          PAID|   2025-05-04|
|ORD100008|   Delivered|          PAID|   2025-08-05|
|ORD100009|   Delivered|          PAID|   2025-11-23|
|ORD100010|   Delivered|          PAID|   2025-11-08|
|ORD100011|   Confirmed|          PAID|         NULL|
|ORD100012|    Returned|      REFUNDED|   2025-11-05|
|ORD100013|   Delivered|          PAID|   2025-08-17|
|ORD100014|   Delivered|          PAID|   2025-12-31|
|ORD100015|   Delivered|          PAID|   2025-01-09|
|ORD100016|   Confirmed|          PAID|         NULL|
|ORD100017|   Delivered|    

In [0]:
spark.sql("""
          select order_id,
          count(*) as record_count
          from workspace.ecommerce_project.silver_orders
          group by order_id
          having count(*)>1
          """).show()

+--------+------------+
|order_id|record_count|
+--------+------------+
+--------+------------+



In [0]:
(
    silver_target.alias("target")
    .merge(
        incremental_silver_orders.alias("source"),
        "target.order_id = source.order_id"
    )
    .whenMatchedDelete(
        condition="source.delte_flag = true"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute
)

<bound method DeltaMergeBuilder.execute of <delta.connect.tables.DeltaMergeBuilder object at 0xfff947199df0>>